# Full Classification Benchmark

**Classifiers:** KNN, Linear SVM, RBF SVM (C=0.5/1/2), Random Forest, Ward-linkage, MLP

**PCA Dimensions:** 10, 20, 50, 100, None

**Descriptors:** Ord_PI, Inter_PI, 3D_PI, Sixpack_Rips, Sixpack_Chroma, Inter+Ord, 3D+Ord

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, glob, numpy as np, pandas as pd, warnings, time
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import accuracy_score, f1_score
from scipy.optimize import linear_sum_assignment
warnings.filterwarnings('ignore')

BASE = '/content/drive/MyDrive/URP'
VECTOR_DIR = os.path.join(BASE, 'Final_Vector')
print(f'Vector dir: {VECTOR_DIR}')

## 1. Ground Truth & Soft Accuracy

In [ ]:
M1=[[0,0,1,1,1,1,1,1],[0,0,1,1,1,1,1,1],[2,2,3,3,3,3,3,3],[2,2,3,3,3,3,3,3],[2,2,3,3,3,3,3,3],[2,2,3,3,3,3,3,3],[2,2,3,3,3,3,3,3],[2,2,3,3,3,3,3,3]]
M2=[[0,0,1,1,1,1,1,1],[0,0,1,1,1,1,1,1],[2,2,3,3,3,3,3,3],[2,2,3,3,3,3,3,3],[2,2,3,3,3,3,3,4],[2,2,3,3,3,3,3,3],[2,2,3,3,3,3,4,4],[2,2,3,3,3,3,3,3]]
M3=[[6,6,7,7,7,7,7,7],[6,6,6,7,7,7,7,7],[9,6,3,3,3,3,3,3],[9,10,3,4,4,3,3,4],[9,10,3,3,4,4,3,4],[9,10,3,4,4,4,4,4],[9,10,3,4,3,4,4,4],[9,10,3,4,3,4,4,4]]
M4=[[6,6,12,12,7,7,7,7],[6,6,12,12,7,7,7,7],[9,6,6,11,7,7,4,4],[9,9,6,3,3,4,4,4],[9,9,10,3,3,4,4,4],[9,9,10,3,3,4,4,4],[9,9,10,4,4,4,4,4],[9,9,10,4,4,4,4,4]]
M5=[[6,6,12,12,12,12,7,7],[6,6,12,12,12,12,12,7],[9,9,6,11,11,11,12,11],[9,9,6,11,11,11,4,4],[9,9,13,13,4,4,4,4],[9,9,13,10,4,4,4,4],[9,9,13,10,4,4,4,4],[9,9,10,10,4,4,4,4]]
M6=[[6,12,12,12,12,12,12,12],[6,6,12,12,12,12,12,12],[9,6,6,11,11,11,11,11],[9,9,6,11,11,11,11,11],[9,9,6,6,6,13,4,4],[9,9,6,13,13,4,4,4],[9,9,6,13,4,4,4,4],[9,9,6,13,4,4,4,4]]
M7=[[6,6,12,12,12,12,12,12],[9,6,12,12,12,12,12,12],[9,6,6,11,11,11,11,12],[9,6,6,11,11,11,11,11],[9,9,6,6,11,11,11,11],[9,9,6,6,11,11,11,4],[9,9,6,6,13,13,4,4],[9,9,6,13,13,4,4,4]]
M8=[[6,12,12,12,12,12,12,12],[6,6,12,12,12,12,12,12],[9,6,6,6,11,11,11,11],[9,6,6,6,11,11,11,11],[9,9,6,6,11,11,11,11],[9,9,6,6,6,11,11,11],[9,9,6,6,13,13,11,11],[9,9,6,6,13,13,11,4]]
GT = np.asarray([M1,M2,M3,M4,M5,M6,M7,M8])

def get_label(task_id):
    idx = task_id - 1
    return GT[idx%64//8][idx//64][idx%8]

# Adjacent phases
def get_adjacent(matrices):
    adj = set()
    for M in matrices:
        M = np.array(M)
        r,c = M.shape
        for i in range(r):
            for j in range(c):
                for di,dj in [(-1,0),(1,0),(0,-1),(0,1)]:
                    ni,nj = i+di, j+dj
                    if 0<=ni<r and 0<=nj<c and M[i,j]!=M[ni,nj]:
                        adj.add(tuple(sorted([int(M[i,j]),int(M[ni,nj])])))
    d = {}
    for p1,p2 in adj:
        d.setdefault(p1,[]).append(p2)
        d.setdefault(p2,[]).append(p1)
    return d

ADJ = get_adjacent(GT)
ALL_CLASSES = sorted(np.unique(GT))
N_CLASSES = len(ALL_CLASSES)

def soft_accuracy(y_true, y_pred):
    n = len(y_true)
    correct = sum(1 for t,p in zip(y_true,y_pred)
                  if t==p or (t in ADJ and p in ADJ[t]))
    return correct/n if n>0 else 0.0

print(f'{N_CLASSES} classes, {len(ADJ)} adjacent entries')

## 2. 데이터 로딩

In [ ]:
def load_pi(data_dir, prefix):
    files = sorted(glob.glob(os.path.join(data_dir, f'{prefix}_*.npz')))
    X_list, y_list = [], []
    for fp in files:
        try:
            sim_idx = int(os.path.basename(fp).split('_')[-1].split('.')[0])
            label = get_label(sim_idx)
            data = np.load(fp, allow_pickle=True)
            features = []
            for key in sorted(data.keys()):
                arr = data[key]
                if hasattr(arr,'item') and arr.ndim==0: arr = arr.item()
                if isinstance(arr, dict):
                    for k in sorted(arr.keys()):
                        val = arr[k]
                        if isinstance(val, dict):
                            for dk in sorted(val.keys()): features.extend(np.asarray(val[dk]).flatten())
                        else: features.extend(np.asarray(val).flatten())
                else: features.extend(np.asarray(arr).flatten())
            X_list.append(features); y_list.append(label)
        except Exception as e:
            print(f'  Skip {fp}: {e}')
    if not X_list: return None, None
    return np.nan_to_num(np.array(X_list, dtype=float)), np.array(y_list)

datasets = {}
for name in ['Inter_PI', '3D_PI', 'Ord_PI', 'Sixpack_Rips', 'Sixpack_Chroma']:
    path = os.path.join(VECTOR_DIR, name)
    if os.path.exists(path):
        print(f'Loading {name}...', end=' ')
        X, y = load_pi(path, name)
        if X is not None:
            datasets[name] = {'X': X, 'y': y}
            print(f'{X.shape}')
        else: print('EMPTY')

# 결합 벡터
if 'Inter_PI' in datasets and 'Ord_PI' in datasets:
    datasets['Inter+Ord'] = {
        'X': np.hstack([datasets['Inter_PI']['X'], datasets['Ord_PI']['X']]),
        'y': datasets['Inter_PI']['y']}
    print(f'Inter+Ord: {datasets["Inter+Ord"]["X"].shape}')
if '3D_PI' in datasets and 'Ord_PI' in datasets:
    datasets['3D+Ord'] = {
        'X': np.hstack([datasets['3D_PI']['X'], datasets['Ord_PI']['X']]),
        'y': datasets['3D_PI']['y']}
    print(f'3D+Ord: {datasets["3D+Ord"]["X"].shape}')

METHODS = [m for m in ['Ord_PI','Inter_PI','3D_PI','Sixpack_Rips','Sixpack_Chroma','Inter+Ord','3D+Ord'] if m in datasets]
print(f'\nLoaded: {METHODS}')

## 3. Classifier 정의

In [ ]:
PCA_DIMS = [10, 20, 50, 100, None]
N_SPLITS = 5
RS = 42

def get_classifiers():
    return {
        'KNN(5)':       KNeighborsClassifier(n_neighbors=5),
        'LinearSVM':    SVC(kernel='linear', C=1.0),
        'RBF_C0.5':     SVC(kernel='rbf', C=0.5),
        'RBF_C1.0':     SVC(kernel='rbf', C=1.0),
        'RBF_C2.0':     SVC(kernel='rbf', C=2.0),
        'RF(200)':      RandomForestClassifier(n_estimators=200, random_state=RS),
        'MLP':          MLPClassifier(hidden_layer_sizes=(128,64), max_iter=500,
                                      early_stopping=True, random_state=RS),
    }

# Ward-linkage (비지도 → Hungarian 매핑)
def ward_classify(X, y, n_clusters):
    ward = AgglomerativeClustering(n_clusters=n_clusters, linkage='ward')
    cl = ward.fit_predict(X)
    # Hungarian mapping
    ucl, ucs = np.unique(cl), np.unique(y)
    size = max(len(ucl), len(ucs))
    cost = np.zeros((size, size))
    for i, c in enumerate(ucl):
        mask = cl == c
        for j, cs in enumerate(ucs):
            cost[i,j] = np.sum(mask) - np.sum(y[mask] == cs)
    ri, ci = linear_sum_assignment(cost)
    mapping = {ucl[r]: ucs[c] for r,c in zip(ri,ci) if r<len(ucl) and c<len(ucs)}
    y_pred = np.array([mapping.get(c, -1) for c in cl])
    return y_pred

print(f'Classifiers: {list(get_classifiers().keys())} + Ward-linkage')
print(f'PCA dims: {PCA_DIMS}')

## 4. 전체 평가 실행

In [ ]:
results = []
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RS)
total_runs = len(METHODS) * len(PCA_DIMS) * (len(get_classifiers()) + 1)  # +1 for Ward
run_count = 0
t_start = time.time()

for method in METHODS:
    X_raw = datasets[method]['X']
    y = datasets[method]['y']

    for pca_dim in PCA_DIMS:
        label_dim = f'PCA{pca_dim}' if pca_dim else 'NoPCA'

        # Preprocess
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X_raw)
        if pca_dim is not None and pca_dim < X_scaled.shape[1]:
            X = PCA(n_components=pca_dim, random_state=RS).fit_transform(X_scaled)
        else:
            X = X_scaled

        # ---- Supervised classifiers (5-fold CV) ----
        for clf_name, clf_template in get_classifiers().items():
            fold_soft, fold_strict, fold_f1 = [], [], []
            for tr_idx, te_idx in skf.split(X, y):
                clf = clf_template.__class__(**clf_template.get_params())
                clf.fit(X[tr_idx], y[tr_idx])
                yp = clf.predict(X[te_idx])
                fold_soft.append(soft_accuracy(y[te_idx], yp))
                fold_strict.append(accuracy_score(y[te_idx], yp))
                fold_f1.append(f1_score(y[te_idx], yp, average='macro', zero_division=0))

            results.append({
                'Method': method, 'PCA': label_dim, 'Classifier': clf_name,
                'Soft(%)': np.mean(fold_soft)*100,
                'Strict(%)': np.mean(fold_strict)*100,
                'F1(%)': np.mean(fold_f1)*100,
            })
            run_count += 1

        # ---- Ward-linkage (비지도, 전체 데이터) ----
        yp_ward = ward_classify(X, y, N_CLASSES)
        results.append({
            'Method': method, 'PCA': label_dim, 'Classifier': 'Ward',
            'Soft(%)': soft_accuracy(y, yp_ward)*100,
            'Strict(%)': accuracy_score(y, yp_ward)*100,
            'F1(%)': f1_score(y, yp_ward, average='macro', zero_division=0)*100,
        })
        run_count += 1

        elapsed = time.time() - t_start
        eta = elapsed/run_count*(total_runs-run_count)/60
        print(f'[{run_count:>4d}/{total_runs}] {method} / {label_dim} done (ETA {eta:.0f}min)')

print(f'\nTotal: {(time.time()-t_start)/60:.1f}min')

## 5. 결과 테이블

In [ ]:
df = pd.DataFrame(results)

# 전체 결과 저장
out_path = os.path.join(BASE, 'Final_Results')
os.makedirs(out_path, exist_ok=True)
df.to_csv(os.path.join(out_path, 'full_benchmark.csv'), index=False)
print(f'Saved to {out_path}/full_benchmark.csv\n')

# Best per method × PCA
print('=== Best Classifier per Method × PCA ===')
pivot = df.loc[df.groupby(['Method','PCA'])['Soft(%)'].idxmax()]
pivot = pivot[['Method','PCA','Classifier','Soft(%)','Strict(%)','F1(%)']]
pivot = pivot.sort_values(['Method','PCA']).reset_index(drop=True)
print(pivot.to_string(index=False))

# Best overall per method
print('\n=== Best Overall per Method ===')
best = df.loc[df.groupby('Method')['Soft(%)'].idxmax()]
best = best[['Method','PCA','Classifier','Soft(%)','Strict(%)','F1(%)']]
print(best.to_string(index=False))

In [ ]:
# Heatmap: Method × Classifier (best PCA)
import matplotlib.pyplot as plt
import seaborn as sns

pivot_heat = df.groupby(['Method','Classifier'])['Soft(%)'].max().unstack()
fig, ax = plt.subplots(figsize=(12, 5))
sns.heatmap(pivot_heat, annot=True, fmt='.1f', cmap='YlOrRd', ax=ax)
ax.set_title('Soft Accuracy (%) — Best PCA per cell')
plt.tight_layout()
plt.savefig(os.path.join(out_path, 'heatmap_method_clf.png'), dpi=150)
plt.show()

# Heatmap: Method × PCA (best classifier)
pivot_pca = df.groupby(['Method','PCA'])['Soft(%)'].max().unstack()
fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(pivot_pca, annot=True, fmt='.1f', cmap='YlOrRd', ax=ax)
ax.set_title('Soft Accuracy (%) — Best Classifier per cell')
plt.tight_layout()
plt.savefig(os.path.join(out_path, 'heatmap_method_pca.png'), dpi=150)
plt.show()